<a href="https://colab.research.google.com/github/ronniewillaert/SPM-Textbook-Python/blob/main/notebooks/part-01-foundations/ch07_single_molecule_force_spectroscopy/SPM_Ch7_Single_Molecule_Force_Spectroscopy_Python_Exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 7 — Single-Molecule Force Spectroscopy: Python Exercises

**SPM Syllabus 2026** · Interactive simulations for AFM-based single-molecule force spectroscopy (SMFS)

These exercises accompany Chapter 7 of the textbook. Each exercise builds an interactive simulation
that lets you explore how the basic models of SMFS — the **worm-like chain (WLC)** for polymer
stretching, the **Bell–Evans framework** for force-driven dissociation, and the **statistical
machinery** that turns a collection of noisy rupture events into kinetic parameters — combine to
produce the experimental signatures you see at the AFM.

The five exercises follow the chapter:

| # | Topic | Sections |
|---|-------|----------|
| 1 | Worm-like chain (WLC) analysis | 7.5 |
| 2 | Bell–Evans dynamic force spectroscopy | 7.4 |
| 3 | Multiple energy landscapes (single / double barrier / mixtures) | 7.4.6, 7.7.6 |
| 4 | Protein unfolding of modular polyproteins | 7.5.4, 7.6.2 |
| 5 | Statistical filtering and uncertainty in SMFS datasets | 7.7 |

**How to use this notebook.** Run the first two cells (imports and ipywidgets) once. Each exercise
consists of a markdown theory cell followed by an interactive code cell driven by ipywidgets sliders.
Move the sliders, observe the response, and try to predict the effect before letting the simulation update.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy import stats

# Plotting defaults
plt.rcParams.update({
    'figure.dpi': 100,
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'legend.fontsize': 9,
    'lines.linewidth': 2,
})

# Physical constants and unit conventions
KB   = 1.380649e-23    # Boltzmann constant (J/K)
T    = 298.0           # room temperature (K)
kBT  = KB * T          # ~ 4.114 zJ at 298 K, equivalently ~ 4.114 pN.nm
kBT_pNnm = kBT * 1e21  # kBT expressed in pN.nm

NM   = 1e-9            # nanometre  (m)
PN   = 1e-12           # piconewton (N)
NN   = 1e-9            # nanonewton (N)

rng = np.random.default_rng(seed=1)

In [ ]:
try:
    from ipywidgets import interact, FloatSlider, IntSlider, FloatLogSlider, Dropdown, Checkbox
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ipywidgets'])
    from ipywidgets import interact, FloatSlider, IntSlider, FloatLogSlider, Dropdown, Checkbox

---
## 1. Worm-Like Chain (WLC) Analysis

**Sections 7.5.2 – 7.5.6**

The **Marko–Siggia worm-like chain** model describes the entropic elasticity of a semi-flexible
biopolymer. The force $F$ required to extend the polymer to end-to-end distance $x$ is

$$
F(x) \;=\; \frac{k_B T}{L_p}\,
\left[
  \frac{1}{4\,(1 - x/L_c)^2} \;-\; \frac{1}{4} \;+\; \frac{x}{L_c}
\right]
$$

with **persistence length** $L_p$ (intrinsic stiffness scale, ~ 0.4 nm for protein backbone, ~ 50 nm
for double-stranded DNA, ~ 0.4 nm for PEG) and **contour length** $L_c$ (fully extended length of
the chain). The curve has a soft entropic regime at low extension and a steep enthalpic divergence
as $x \to L_c$.

**What you will do.** Generate noisy synthetic WLC data, fit the model, and explore how $L_p$ and
$L_c$ shape the curve. Two modes are available: a **single segment** (one stretched molecule) and
a **multi-segment** mode in which a sequence of WLC segments with increasing $L_c$ models a
polyprotein sawtooth.

In [ ]:
def wlc_force(x, Lp, Lc):
    """Marko-Siggia interpolation formula. x, Lp, Lc all in nm; returns force in pN."""
    # Clip x to avoid divergence at x = Lc
    r = np.clip(x / Lc, 0, 0.995)
    return (kBT_pNnm / Lp) * (0.25 / (1 - r) ** 2 - 0.25 + r)


def interactive_wlc(Lp_nm=0.4, Lc_nm=80.0, noise_pN=5.0, n_points=200,
                    multi_segment=False, n_unfold=4, dLc_nm=28.0):
    """Synthesise WLC data, fit, and plot single- or multi-segment cases."""

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

    if not multi_segment:
        # --- Single WLC segment ---
        x_true = np.linspace(0.05 * Lc_nm, 0.95 * Lc_nm, n_points)
        F_true = wlc_force(x_true, Lp_nm, Lc_nm)
        F_noisy = F_true + rng.normal(0, noise_pN, size=n_points)

        # Fit
        try:
            popt, pcov = curve_fit(wlc_force, x_true, F_noisy,
                                   p0=[0.5, Lc_nm * 1.1],
                                   bounds=([0.05, x_true.max() * 1.02], [10, 1e4]))
            Lp_fit, Lc_fit = popt
            Lp_err, Lc_err = np.sqrt(np.diag(pcov))
        except Exception as e:
            Lp_fit = Lc_fit = np.nan
            Lp_err = Lc_err = np.nan

        ax = axes[0]
        ax.plot(x_true, F_noisy, 'o', ms=2, alpha=0.4, color='steelblue', label='synthetic data')
        ax.plot(x_true, F_true, '-', color='k', lw=1.5, label=f'truth: Lp={Lp_nm} nm, Lc={Lc_nm} nm')
        if not np.isnan(Lp_fit):
            x_fit = np.linspace(x_true.min(), x_true.max(), 300)
            ax.plot(x_fit, wlc_force(x_fit, Lp_fit, Lc_fit), '--', color='crimson', lw=2,
                    label=f'fit: Lp={Lp_fit:.2f}±{Lp_err:.2f} nm, Lc={Lc_fit:.1f}±{Lc_err:.1f} nm')
        ax.set_xlabel('Extension x (nm)')
        ax.set_ylabel('Force (pN)')
        ax.set_title('Single WLC segment')
        ax.set_ylim(0, min(400, F_true.max() * 1.1 + noise_pN * 3))
        ax.legend(loc='upper left')
        ax.grid(alpha=0.3)

        ax2 = axes[1]
        # Sensitivity plot: vary Lp and overlay
        for Lp_alt in [0.3, 0.5, 1.0, 2.0]:
            ax2.plot(x_true, wlc_force(x_true, Lp_alt, Lc_nm), label=f'Lp = {Lp_alt} nm', alpha=0.8)
        ax2.set_xlabel('Extension x (nm)')
        ax2.set_ylabel('Force (pN)')
        ax2.set_title('Effect of persistence length on WLC curve')
        ax2.set_ylim(0, 300)
        ax2.legend()
        ax2.grid(alpha=0.3)

    else:
        # --- Multi-segment sawtooth ---
        Lc_seq = np.array([Lc_nm + i * dLc_nm for i in range(n_unfold + 1)])
        F_peaks_true = []
        ax = axes[0]
        for i, Lc_i in enumerate(Lc_seq):
            x_seg = np.linspace(0.05 * Lc_i, 0.95 * Lc_i, 100)
            F_seg = wlc_force(x_seg, Lp_nm, Lc_i)
            F_seg_noisy = F_seg + rng.normal(0, noise_pN, size=F_seg.size)
            ax.plot(x_seg, F_seg_noisy, '.', ms=2, alpha=0.35, color=f'C{i}')
            ax.plot(x_seg, F_seg, '-', color=f'C{i}', lw=1.5, label=f'Lc = {Lc_i:.1f} nm')
            # 'Rupture' force at 90% extension of the segment
            F_peaks_true.append(wlc_force(0.9 * Lc_i, Lp_nm, Lc_i))
        ax.set_xlabel('Extension x (nm)')
        ax.set_ylabel('Force (pN)')
        ax.set_title(f'Sawtooth: {n_unfold} unfolding events, ΔLc = {dLc_nm} nm')
        ax.set_ylim(0, 350)
        ax.legend(fontsize=8, loc='upper left')
        ax.grid(alpha=0.3)

        # Right panel: extracted Lc per peak
        ax2 = axes[1]
        peak_idx = np.arange(1, n_unfold + 2)
        ax2.plot(peak_idx, Lc_seq, 'o-', color='crimson', label='Lc per segment')
        ax2.set_xlabel('Segment index')
        ax2.set_ylabel('Contour length Lc (nm)')
        ax2.set_title(f'ΔLc between segments = {dLc_nm:.1f} nm')
        ax2.grid(alpha=0.3)
        ax2.legend()

    plt.tight_layout()
    plt.show()


interact(
    interactive_wlc,
    Lp_nm=FloatSlider(min=0.3, max=5.0, step=0.05, value=0.4, description='Lp (nm)'),
    Lc_nm=FloatSlider(min=20, max=300, step=5, value=80, description='Lc (nm)'),
    noise_pN=FloatSlider(min=0, max=20, step=0.5, value=5, description='noise (pN)'),
    n_points=IntSlider(min=50, max=500, step=50, value=200, description='n points'),
    multi_segment=Checkbox(value=False, description='multi-segment (sawtooth)'),
    n_unfold=IntSlider(min=1, max=8, step=1, value=4, description='# unfold events'),
    dLc_nm=FloatSlider(min=10, max=50, step=1, value=28, description='ΔLc (nm)'),
);

---
## 2. Bell–Evans Dynamic Force Spectroscopy

**Sections 7.4.1 – 7.4.5**

Under a force ramp $F(t) = r \, t$ with constant loading rate $r$ (pN/s), the **Bell–Evans**
model predicts a probability density of rupture force

$$
p(F) \;=\; \frac{k_{\rm off}^0}{r}\,\exp\!\left(\frac{F\,x_\beta}{k_B T}\right)
\;\exp\!\left[\,\frac{k_{\rm off}^0\, k_B T}{r\, x_\beta}
\left(1 - e^{F x_\beta / k_B T}\right)\right]
$$

with **barrier position** $x_\beta$ (Å range for stiff bonds, nm range for soft ones) and
**zero-force off-rate** $k_{\rm off}^0$. The **most probable rupture force** is

$$
F^{*}(r) \;=\; \frac{k_B T}{x_\beta}\,\ln\!\left(\frac{r\, x_\beta}{k_{\rm off}^0\, k_B T}\right)
$$

so a plot of $F^{*}$ vs $\ln(r)$ is linear with slope $k_BT / x_\beta$ and intercept controlled
by $k_{\rm off}^0$.

**What you will do.** Generate ensembles of simulated rupture events for several loading rates,
build histograms, extract $F^{*}(r)$, and reconstruct $x_\beta$ and $k_{\rm off}^0$ from the DFS plot.

In [ ]:
def sample_bell_evans(r_pN_per_s, x_beta_nm, k_off0_s, n_events=2000, rng=rng):
    """Inverse-transform sampling of the Bell-Evans rupture-force distribution.

    r_pN_per_s : loading rate (pN/s)
    x_beta_nm  : barrier position (nm)
    k_off0_s   : zero-force off-rate (1/s)
    """
    # Cumulative survival probability: S(F) = exp[ (k_off0 kBT / r x_b) (1 - exp(F x_b / kBT)) ]
    # Force a random survival value u ∈ (0,1):
    u = rng.uniform(0, 1, size=n_events)
    a = k_off0_s * kBT_pNnm / (r_pN_per_s * x_beta_nm)
    F = (kBT_pNnm / x_beta_nm) * np.log(1 - np.log(u) / a)
    return F  # pN


def F_star_bell_evans(r_pN_per_s, x_beta_nm, k_off0_s):
    return (kBT_pNnm / x_beta_nm) * np.log(r_pN_per_s * x_beta_nm /
                                           (k_off0_s * kBT_pNnm))


def interactive_bell_evans(x_beta_nm=0.30, k_off0_s=0.10, noise_pN=0.0, n_events=2000,
                          add_noise=False):
    r_values = np.array([10, 100, 1000, 10000, 100000])  # pN/s

    fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.3))

    # ---- Histograms ----
    ax = axes[0]
    F_star_meas = []
    colors = plt.cm.viridis(np.linspace(0.15, 0.9, len(r_values)))
    for r, c in zip(r_values, colors):
        F = sample_bell_evans(r, x_beta_nm, k_off0_s, n_events=n_events)
        if add_noise:
            F = F + rng.normal(0, noise_pN, size=F.size)
        bins = np.linspace(0, max(400, F.max() * 1.05), 50)
        h, edges = np.histogram(F, bins=bins)
        centres = 0.5 * (edges[:-1] + edges[1:])
        ax.plot(centres, h / h.sum(), color=c, lw=1.5,
                label=f'r = {r:g} pN/s')
        F_star_meas.append(centres[np.argmax(h)])
    ax.set_xlabel('Rupture force F (pN)')
    ax.set_ylabel('Probability (normalised)')
    ax.set_title('Rupture-force histograms across loading rates')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

    # ---- DFS plot ----
    ax2 = axes[1]
    F_star_meas = np.array(F_star_meas)
    F_star_theory = F_star_bell_evans(r_values, x_beta_nm, k_off0_s)
    ax2.semilogx(r_values, F_star_meas, 'o', ms=8, color='crimson', label='from histograms')
    ax2.semilogx(r_values, F_star_theory, '-', color='k', alpha=0.5,
                 label='Bell–Evans theory')

    # Linear fit: F* = (kBT/x_b) ln(r) + const
    slope, intercept, *_ = stats.linregress(np.log(r_values), F_star_meas)
    x_beta_fit = kBT_pNnm / slope
    k_off0_fit = (slope / x_beta_fit) * np.exp(-intercept / slope) if False else (
        np.exp(-intercept / slope) * slope / x_beta_fit
    )
    # Simpler back-calc of k_off0:
    # F* = (kBT/x_b) ln(r x_b / (k_off0 kBT))
    # => intercept_natural = (kBT/x_b) ln(x_b/(k_off0 kBT))
    # => k_off0 = x_b/(kBT) * exp(-intercept/(kBT/x_b))
    k_off0_fit = (x_beta_fit / kBT_pNnm) * np.exp(-intercept * x_beta_fit / kBT_pNnm)

    r_fine = np.logspace(np.log10(r_values.min()/2), np.log10(r_values.max()*2), 200)
    ax2.semilogx(r_fine, slope * np.log(r_fine) + intercept, '--',
                 color='C0', lw=1.5,
                 label=f'fit: xβ={x_beta_fit:.2f} nm, k_off0={k_off0_fit:.2g} /s')
    ax2.set_xlabel('Loading rate r (pN/s)')
    ax2.set_ylabel('Most probable force F* (pN)')
    ax2.set_title(f'DFS plot — truth: xβ={x_beta_nm} nm, k_off0={k_off0_s:.2g} /s')
    ax2.legend(fontsize=8)
    ax2.grid(alpha=0.3, which='both')

    plt.tight_layout()
    plt.show()


interact(
    interactive_bell_evans,
    x_beta_nm=FloatSlider(min=0.05, max=1.0, step=0.02, value=0.30, description='xβ (nm)'),
    k_off0_s=FloatLogSlider(min=-4, max=2, step=0.1, value=0.1, description='k_off0 (1/s)'),
    noise_pN=FloatSlider(min=0, max=30, step=1, value=10, description='σ_noise (pN)'),
    n_events=IntSlider(min=200, max=5000, step=200, value=2000, description='# events'),
    add_noise=Checkbox(value=False, description='add instrument noise'),
);

---
## 3. Multiple Energy Landscapes

**Sections 7.4.6 and 7.7.6**

When a molecule has more than one barrier — for example an **inner** and an **outer** barrier along
the same coordinate — the Bell–Evans escape rate becomes a sum of two contributions, and the DFS plot
develops a **kink**: at low loading rates the outer (longer-lived) barrier dominates, at high loading
rates the inner one does.

A different scenario — but with a similar histogram signature — is a **mixture of two molecular
species** with different $(x_\beta, k_{\rm off}^0)$. The histogram will be bimodal; the DFS plot may
look like two parallel lines.

**What you will do.** Run Monte–Carlo rupture simulations for the three cases (single barrier, two
sequential barriers, two-species mixture). Compare the histograms and DFS plots, fit a naive
single-barrier model to a multi-pathway dataset, and observe the systematic error it produces.

In [ ]:
def sample_two_barrier(r_pN_per_s, x1_nm, k1_s, x2_nm, k2_s, n_events=2000, rng=rng):
    """Two parallel Bell-Evans escape pathways. Total survival is the product of two,
    so we sample the first-passage of whichever barrier loses first."""
    F1 = sample_bell_evans(r_pN_per_s, x1_nm, k1_s, n_events=n_events, rng=rng)
    F2 = sample_bell_evans(r_pN_per_s, x2_nm, k2_s, n_events=n_events, rng=rng)
    return np.minimum(F1, F2)


def sample_mixture(r_pN_per_s, x1_nm, k1_s, x2_nm, k2_s, fraction_A=0.5,
                   n_events=2000, rng=rng):
    """Two independent molecular species in the dataset."""
    nA = int(round(fraction_A * n_events))
    nB = n_events - nA
    F_A = sample_bell_evans(r_pN_per_s, x1_nm, k1_s, n_events=nA, rng=rng)
    F_B = sample_bell_evans(r_pN_per_s, x2_nm, k2_s, n_events=nB, rng=rng)
    return np.concatenate([F_A, F_B])


def interactive_multibarrier(scenario='single barrier',
                             x_inner=0.15, k_inner=1e-2,
                             x_outer=0.50, k_outer=1e-1,
                             fraction_A=0.5, n_events=2000):
    r_values = np.logspace(1, 5, 5)  # 10 .. 1e5 pN/s

    fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.3))
    ax_hist, ax_dfs = axes
    colors = plt.cm.viridis(np.linspace(0.15, 0.9, len(r_values)))
    F_star_meas = []

    for r, c in zip(r_values, colors):
        if scenario == 'single barrier':
            F = sample_bell_evans(r, x_outer, k_outer, n_events=n_events)
        elif scenario == 'double barrier (nested)':
            F = sample_two_barrier(r, x_inner, k_inner, x_outer, k_outer, n_events=n_events)
        else:  # mixture
            F = sample_mixture(r, x_inner, k_inner, x_outer, k_outer,
                               fraction_A=fraction_A, n_events=n_events)
        bins = np.linspace(0, max(400, F.max() * 1.05), 60)
        h, edges = np.histogram(F, bins=bins)
        centres = 0.5 * (edges[:-1] + edges[1:])
        ax_hist.plot(centres, h / h.sum(), color=c, lw=1.4, label=f'r = {r:.0g}')
        F_star_meas.append(centres[np.argmax(h)])

    ax_hist.set_xlabel('Rupture force F (pN)')
    ax_hist.set_ylabel('Probability')
    ax_hist.set_title(f'Histograms — scenario: {scenario}')
    ax_hist.legend(fontsize=8, title='pN/s')
    ax_hist.grid(alpha=0.3)

    # DFS plot with naive single-barrier fit
    F_star_meas = np.array(F_star_meas)
    slope, intercept, r_corr, *_ = stats.linregress(np.log(r_values), F_star_meas)
    x_beta_fit = kBT_pNnm / slope
    k_off0_fit = (x_beta_fit / kBT_pNnm) * np.exp(-intercept * x_beta_fit / kBT_pNnm)
    ax_dfs.semilogx(r_values, F_star_meas, 'o', ms=8, color='crimson')
    r_fine = np.logspace(np.log10(r_values.min()/2), np.log10(r_values.max()*2), 200)
    ax_dfs.semilogx(r_fine, slope * np.log(r_fine) + intercept, '--', color='C0',
                    label=f'naive 1-barrier fit\nxβ={x_beta_fit:.2f} nm\nk_off0={k_off0_fit:.2g}/s')
    ax_dfs.set_xlabel('Loading rate r (pN/s)')
    ax_dfs.set_ylabel('F* (pN)')
    ax_dfs.set_title('DFS plot — does a single line really fit?')
    ax_dfs.legend(fontsize=8, loc='lower right')
    ax_dfs.grid(alpha=0.3, which='both')

    plt.tight_layout()
    plt.show()


interact(
    interactive_multibarrier,
    scenario=Dropdown(
        options=['single barrier', 'double barrier (nested)', 'mixture (two species)'],
        value='single barrier', description='scenario'),
    x_inner=FloatSlider(min=0.05, max=0.6, step=0.02, value=0.15, description='x_inner (nm)'),
    k_inner=FloatLogSlider(min=-4, max=2, step=0.1, value=1e-2, description='k_inner (1/s)'),
    x_outer=FloatSlider(min=0.10, max=1.5, step=0.02, value=0.50, description='x_outer (nm)'),
    k_outer=FloatLogSlider(min=-4, max=2, step=0.1, value=1e-1, description='k_outer (1/s)'),
    fraction_A=FloatSlider(min=0.05, max=0.95, step=0.05, value=0.5, description='fraction A'),
    n_events=IntSlider(min=200, max=5000, step=200, value=2000, description='# events'),
);

---
## 4. Protein Unfolding Simulations

**Sections 7.5.4 and 7.6.2**

A modular polyprotein (e.g. eight identical Ig domains of titin) under AFM pulling produces the iconic
**sawtooth** force trace. We combine two ingredients to simulate it from scratch:

1. **WLC stretching** of the unfolded chain between domain unfolding events, with contour length
$L_c^{(n)} = L_c^{(0)} + n\, \Delta L_c$ after $n$ domains have unfolded.
2. **Bell–Evans unfolding kinetics** of each remaining folded domain, with the per-domain unfolding
rate $k_u(F) = k_u^0 \exp(F x_\beta / k_B T)$.

We integrate the force build-up at constant pulling velocity $v$ and use the cumulative survival
probability $S(t) = \exp\!\left[-N_{\rm folded}\int_0^t k_u(F(t'))\,dt'\right]$ to draw the next
unfolding event by inverse-transform sampling. After each unfolding the contour length increases
by $\Delta L_c$, the WLC relaxes, the cantilever rebuilds force, and the process repeats.

In [ ]:
def simulate_polyprotein(N_domains=8, Lc0_nm=20.0, dLc_nm=28.0, Lp_nm=0.4,
                         v_nm_per_s=1000.0, k_u0_s=1e-4, x_beta_nm=0.25,
                         k_cant_pN_per_nm=15.0, noise_pN=2.0, rng=rng):
    """Stochastic simulation of constant-velocity AFM unfolding of a modular polyprotein.

    Returns x (nm), F (pN), and the list of unfolding (x, F, ΔLc) events."""

    x_step_nm = 0.5  # piezo step
    z_piezo_max = (Lc0_nm + N_domains * dLc_nm) * 1.05
    z_piezo = 0.0
    Lc = Lc0_nm
    folded = N_domains
    x_handle = 0.0  # how far the polymer is stretched, end-to-end
    xs, Fs = [], []
    events = []

    while z_piezo < z_piezo_max:
        # The molecule stretches; the cantilever bends by (z_piezo - x_handle).
        # Force balance: cantilever stiffness * (z_piezo - x_handle) = WLC force at x_handle
        # We invert numerically by bisecting on x_handle:
        # Given current z_piezo and Lc, find x_handle that satisfies the equality.
        # We approximate: x_handle = z_piezo - F / k_cant ; iterate.
        x_h = min(x_handle, 0.95 * Lc)
        for _ in range(20):
            F = wlc_force(x_h, Lp_nm, Lc)
            x_h_new = z_piezo - F / k_cant_pN_per_nm
            x_h_new = min(max(0, x_h_new), 0.99 * Lc)
            if abs(x_h_new - x_h) < 0.005:
                break
            x_h = x_h_new
        x_handle = x_h
        F = wlc_force(x_handle, Lp_nm, Lc)
        # Probability of an unfolding event during this piezo step
        if folded > 0:
            dt = x_step_nm / v_nm_per_s
            k_u = k_u0_s * np.exp(F * x_beta_nm / kBT_pNnm)
            p_unfold = 1.0 - np.exp(-folded * k_u * dt)
            if rng.random() < p_unfold:
                events.append((x_handle, F, dLc_nm))
                folded -= 1
                Lc += dLc_nm
                # Force drops because the chain instantly elongates; x_handle stays put,
                # but the force at the same x with the new Lc is much smaller.
        Fs.append(F + rng.normal(0, noise_pN))
        xs.append(x_handle)
        z_piezo += x_step_nm
        # End if WLC force diverges and last domain unfolded ->
        if folded == 0 and x_handle > 0.95 * Lc:
            # Add a final 'detachment' point
            break
    return np.array(xs), np.array(Fs), events


def interactive_polyprotein(N_domains=8, dLc_nm=28.0, Lp_nm=0.4,
                            v_nm_per_s=1000.0, k_u0_s=1e-4, x_beta_nm=0.25,
                            noise_pN=2.0, seed=0):
    sim_rng = np.random.default_rng(seed)
    xs, Fs, events = simulate_polyprotein(
        N_domains=N_domains, dLc_nm=dLc_nm, Lp_nm=Lp_nm,
        v_nm_per_s=v_nm_per_s, k_u0_s=k_u0_s, x_beta_nm=x_beta_nm,
        noise_pN=noise_pN, rng=sim_rng,
    )

    fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.3))
    ax = axes[0]
    ax.plot(xs, Fs, '-', color='steelblue', lw=1.2, alpha=0.8)
    if events:
        xs_e, Fs_e, _ = zip(*events)
        ax.plot(xs_e, Fs_e, 'o', color='crimson', ms=8, label=f'{len(events)} unfolding peaks')
    ax.set_xlabel('Extension x (nm)')
    ax.set_ylabel('Force (pN)')
    ax.set_title(f'Polyprotein sawtooth — v = {v_nm_per_s:g} nm/s')
    ax.set_ylim(0, max(300, Fs.max() * 1.1 if len(Fs) else 300))
    ax.grid(alpha=0.3)
    ax.legend()

    # Distribution of unfolding forces over multiple repeats
    ax2 = axes[1]
    n_repeats = 30
    all_F_u = []
    for s in range(n_repeats):
        _, _, ev = simulate_polyprotein(
            N_domains=N_domains, dLc_nm=dLc_nm, Lp_nm=Lp_nm,
            v_nm_per_s=v_nm_per_s, k_u0_s=k_u0_s, x_beta_nm=x_beta_nm,
            noise_pN=noise_pN, rng=np.random.default_rng(seed + 100 + s),
        )
        if ev:
            all_F_u.extend([f for _, f, _ in ev])
    if all_F_u:
        ax2.hist(all_F_u, bins=20, color='C2', alpha=0.7, edgecolor='k')
        ax2.axvline(np.mean(all_F_u), color='crimson', lw=2,
                    label=f'⟨F_u⟩ = {np.mean(all_F_u):.0f} pN')
    ax2.set_xlabel('Unfolding force F_u (pN)')
    ax2.set_ylabel('Count')
    ax2.set_title(f'Unfolding-force distribution ({n_repeats} repeats)')
    ax2.legend()
    ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()


interact(
    interactive_polyprotein,
    N_domains=IntSlider(min=2, max=10, step=1, value=8, description='N domains'),
    dLc_nm=FloatSlider(min=10, max=40, step=1, value=28, description='ΔLc (nm)'),
    Lp_nm=FloatSlider(min=0.3, max=1.0, step=0.05, value=0.4, description='Lp (nm)'),
    v_nm_per_s=FloatLogSlider(min=2, max=4, step=0.1, value=1000, description='v (nm/s)'),
    k_u0_s=FloatLogSlider(min=-6, max=-2, step=0.2, value=1e-4, description='k_u0 (1/s)'),
    x_beta_nm=FloatSlider(min=0.1, max=0.5, step=0.02, value=0.25, description='xβ (nm)'),
    noise_pN=FloatSlider(min=0, max=15, step=1, value=2, description='σ_noise (pN)'),
    seed=IntSlider(min=0, max=20, step=1, value=0, description='seed'),
);

---
## 5. Statistical Filtering and Uncertainty

**Section 7.7**

Real SMFS experiments rarely deliver "clean" rupture-force distributions. The raw collection of
rupture events is a mixture of

- **specific** ruptures from the molecule of interest,
- **nonspecific adhesion** at short tip–surface separations,
- a **minority population** (e.g. a different conformer, double-bond events, dimers),
- **electronic / thermal noise** that fattens the distribution.

The challenge is that **filtering changes the answer**. Common filters in the literature include:
low-force cut-off, linker-length window (only keep events that ruptured at distances consistent
with the linker contour length), and minimum tip–surface separation to exclude pure adhesion.

**What you will do.** Generate a realistic mixed dataset, apply three filtering strategies, and
compare the rupture-force histograms and the Bell–Evans parameters fitted from the filtered data.
Use bootstrap resampling to attach a 95 % confidence interval to $F^{*}$ and to the DFS slope.

In [ ]:
def make_mixed_dataset(n_events=2000, frac_nonspecific=0.20, frac_minority=0.10,
                      r_pN_per_s=1000,
                      x_beta_main=0.30, k_off0_main=0.1,
                      x_beta_min=0.7, k_off0_min=10,
                      noise_pN=8.0, rng=rng):
    """Returns rupture forces (pN) and the corresponding tip-surface separation (nm).
    Each event is labelled 'specific', 'minority', or 'nonspecific'."""

    n_ns = int(round(frac_nonspecific * n_events))
    n_min = int(round(frac_minority * n_events))
    n_main = n_events - n_ns - n_min

    # Specific events: Bell-Evans + linker-WLC distance
    F_main = sample_bell_evans(r_pN_per_s, x_beta_main, k_off0_main, n_events=n_main, rng=rng)
    d_main = rng.normal(30, 4, size=n_main)  # typical PEG linker length

    # Minority population: shorter-lived, longer x_beta
    F_min = sample_bell_evans(r_pN_per_s, x_beta_min, k_off0_min, n_events=n_min, rng=rng)
    d_min = rng.normal(30, 4, size=n_min)

    # Nonspecific adhesion: large random forces at small tip-surface distances
    F_ns = np.abs(rng.normal(50, 25, size=n_ns))
    d_ns = rng.exponential(5, size=n_ns)  # short, exponential tail

    F_all = np.concatenate([F_main, F_min, F_ns]) + rng.normal(0, noise_pN, size=n_events)
    d_all = np.concatenate([d_main, d_min, d_ns])
    labels = np.array(['specific'] * n_main + ['minority'] * n_min + ['nonspecific'] * n_ns)

    # Shuffle
    idx = rng.permutation(n_events)
    return F_all[idx], d_all[idx], labels[idx]


def bootstrap_F_star(F, n_boot=500, rng=rng):
    """Bootstrap CI for the histogram maximum F* using KDE bandwidth = 5 pN."""
    F_stars = []
    for _ in range(n_boot):
        sample = rng.choice(F, size=F.size, replace=True)
        if sample.size == 0:
            continue
        kde = stats.gaussian_kde(sample, bw_method=0.15)
        grid = np.linspace(0, 400, 400)
        F_stars.append(grid[np.argmax(kde(grid))])
    if not F_stars:
        return np.nan, np.nan, np.nan
    F_stars = np.array(F_stars)
    return np.median(F_stars), np.percentile(F_stars, 2.5), np.percentile(F_stars, 97.5)


def interactive_filtering(frac_nonspecific=0.20, frac_minority=0.10, noise_pN=8,
                          filter_choice='no filter',
                          F_cutoff_pN=20, d_min_nm=20, d_max_nm=45,
                          n_events=2000):
    F, d, labels = make_mixed_dataset(
        n_events=n_events, frac_nonspecific=frac_nonspecific,
        frac_minority=frac_minority, noise_pN=noise_pN,
    )

    # Apply filter
    if filter_choice == 'no filter':
        mask = np.ones_like(F, dtype=bool)
    elif filter_choice == 'low-force cut-off':
        mask = F > F_cutoff_pN
    else:  # linker-length window
        mask = (d > d_min_nm) & (d < d_max_nm)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))

    ax = axes[0]
    bins = np.linspace(0, 300, 60)
    ax.hist(F, bins=bins, color='lightgray', alpha=0.6,
            label=f'raw ({F.size} events)')
    ax.hist(F[mask], bins=bins, color='steelblue', alpha=0.8,
            label=f'after filter ({mask.sum()} events)')
    ax.set_xlabel('Rupture force F (pN)')
    ax.set_ylabel('Count')
    ax.set_title(f'Filtering strategy: {filter_choice}')
    ax.legend()
    ax.grid(alpha=0.3)

    # Bootstrap F* CI
    F_med, F_lo, F_hi = bootstrap_F_star(F[mask], n_boot=300)
    ax.axvline(F_med, color='crimson', lw=2, label='F* (filtered)')
    ax.axvspan(F_lo, F_hi, color='crimson', alpha=0.15, label='95% CI')
    ax.legend(loc='upper right')

    # Composition pie
    ax2 = axes[1]
    raw_counts = np.array([(labels == k).sum() for k in ['specific', 'minority', 'nonspecific']])
    flt_counts = np.array([((labels == k) & mask).sum() for k in ['specific', 'minority', 'nonspecific']])
    w = 0.35
    x_pos = np.arange(3)
    ax2.bar(x_pos - w/2, raw_counts, width=w, color='lightgray', label='raw')
    ax2.bar(x_pos + w/2, flt_counts, width=w, color='steelblue', label='after filter')
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels(['specific', 'minority', 'nonspecific'])
    ax2.set_ylabel('Count')
    ax2.set_title(f'Composition before/after filter\nF* = {F_med:.0f} pN (95% CI: {F_lo:.0f}–{F_hi:.0f} pN)')
    ax2.legend()
    ax2.grid(alpha=0.3, axis='y')

    plt.tight_layout()
    plt.show()


interact(
    interactive_filtering,
    frac_nonspecific=FloatSlider(min=0, max=0.6, step=0.05, value=0.20, description='frac nonspec'),
    frac_minority=FloatSlider(min=0, max=0.5, step=0.05, value=0.10, description='frac minor'),
    noise_pN=FloatSlider(min=0, max=20, step=1, value=8, description='σ_noise (pN)'),
    filter_choice=Dropdown(
        options=['no filter', 'low-force cut-off', 'linker-length window'],
        value='no filter', description='filter'),
    F_cutoff_pN=FloatSlider(min=0, max=80, step=2, value=20, description='F cut (pN)'),
    d_min_nm=FloatSlider(min=5, max=40, step=1, value=20, description='d_min (nm)'),
    d_max_nm=FloatSlider(min=20, max=60, step=1, value=45, description='d_max (nm)'),
    n_events=IntSlider(min=200, max=5000, step=200, value=2000, description='# events'),
);

---

## Summary

| Exercise | Section | Key concept | Take-home |
|---|---|---|---|
| 1 | 7.5 | WLC fit | $L_p$ is constrained by high-force divergence; $L_c$ by low-force regime |
| 2 | 7.4 | Bell–Evans | $F^{*}$ grows linearly with $\ln r$; slope gives $x_\beta$, intercept gives $k_{\rm off}^0$ |
| 3 | 7.4.6, 7.7.6 | Multi-barrier and mixtures | Bimodal histograms and kinks in DFS plots reveal hidden complexity |
| 4 | 7.5.4, 7.6.2 | Polyprotein unfolding | Sawtooth = WLC + Bell–Evans; ΔLc is a structural ruler, F_u depends on $v$ |
| 5 | 7.7 | Filtering and uncertainty | Filtering is part of the measurement: report it as carefully as the result |

### Quick reference — pulling rates and forces in SMFS

| Quantity | Typical AFM range | Meaning |
|---|---|---|
| Pulling velocity $v$ | 10 nm/s – 10 µm/s | Drives the loading rate via $r \approx k\,v$ |
| Loading rate $r$ | 10 pN/s – 100 nN/s | Sets the time available for thermal escape |
| Rupture force $F$ | 20 pN – 1 nN | Depends on $r$, $x_\beta$, $k_{\rm off}^0$ |
| Persistence length $L_p$ | 0.4 nm (protein), 0.4 nm (PEG), 50 nm (dsDNA) | Backbone stiffness scale |
| Barrier position $x_\beta$ | 0.1 nm (covalent) – 1 nm (soft bonds) | Distance to transition state |

### Where to go next

- Replace the synthetic datasets in Exercise 5 with your own AFM SMFS data (CSV import).
- Extend Exercise 3 with **Friddle–Noy–De Yoreo near-equilibrium** corrections to Bell–Evans.
- Couple Exercise 4 to a stiffness-dependent simulation: replace the constant-velocity ramp with
  the actual cantilever response, and compare to your measured curves.